# Layer 2 (FRIENDS), option (b): the paper's Algorithm 2 (warm-up, then eta from the current model)

eta preserves the **new task's head** (KL), computed after 2 warm-up epochs; random noise from epoch 0. Official FRIENDS optimizer
(lr 100, momentum 0.9, 30 steps, lambda 1).

**Done (eps 0.3, official code):** zeta=mu 16/32/64 (BWT -21.4 / -21.4 / -19.9, last 41.3 / 39.6 / 38.8; attack -24.4 / 45.2), friendly-only,
random-only 16-128, clean cost, random-after-warm-up. **To run now:**
1. **eta optimizer settings** (learning rate, momentum) at zeta=mu 32: does freezing pixels at the cap early matter? `b_z32_ref` re-runs the
   old setting on this code version: it should land near -21.4 / 39.6.
2. **BatchNorm control, re-run in this clean folder:** the earlier run gave attack + `nostats` BWT -5.8 but clean + `nostats` -16.6, an odd
   pattern that needs confirming before it is interpreted.

## 1. Settings

In [ ]:
EPS = '0.3'                       # '0.3' = the main table (cautious, eps 0.3); '0.1' = the weaker attack
NOISE = {
    '0.3': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.3/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.3__delta_0.3_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_29.pkl',
    '0.1': '/content/drive/MyDrive/brainwash/brainwash-repro/repro/sweep/noise/cifar100_ewc_cautious_eps0.1/noise_ewc_wcur_1.0_cifar100_ewc_cautious_eps0.1__delta_0.1_dataset_split_cifar100_target_task_0_attacked_task_9_noise_optim_lr_0.005__n_iters_1_n_epochs_5000_seed_0_mode_cautious__w_cur_1.0___min_acc_target_39.pkl',
}
NOISE_PKL = NOISE[EPS]
# One folder per eps, SHARED by the option a / b / c notebooks: A_clean and B_poison then run only once, and each
# notebook's table shows every arm of that eps. Finished arms (a finished log in this folder) are skipped.
RESULTS_DIR = f'/content/drive/MyDrive/brainwash/layer2_runs/layer2_defense_eps{EPS}'
SEED = 0                            # only seed 0: there are no victim checkpoints for other seeds yet

REPO_URL = 'https://github.com/anushkathagle/brainwash-repro.git'
REPO_BRANCH = 'layer2/Friendly'

## 2. GPU and Google Drive

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isfile(NOISE_PKL), f'not found: {NOISE_PKL}'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('noise pkl:', os.path.basename(NOISE_PKL), f'({os.path.getsize(NOISE_PKL)/1e6:.0f} MB)')
print('results  :', RESULTS_DIR)

## 3. Code: the `layer2/Friendly` branch

The Layer-2 code lives in the repo, in `layer2_defense/` (next to `approaches/` and `repro/`), so nothing is written here.
The first run clones the branch; re-running this cell later **pulls the newest commit** of the branch.
Private repo: add a read-only GitHub token as a Colab secret named `GH_TOKEN` (key icon in the left sidebar).

In [ ]:
import subprocess, sys
REPO_DIR = '/content/brainwash-repro'
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
    if tok:
        url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', '-b', REPO_BRANCH, url, REPO_DIR], check=True)
else:   # already cloned in this Colab session: move to the newest commit of the branch
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', url, REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', '-B', REPO_BRANCH, 'FETCH_HEAD'], check=True)
%cd /content/brainwash-repro
assert os.path.isfile('layer2_defense/friends_layer2.py'), 'layer2_defense/ missing: is REPO_BRANCH right?'
sys.path.insert(0, '/content/brainwash-repro/layer2_defense')
import importlib, colab_helpers as H
importlib.reload(H)
H.record_code_version(RESULTS_DIR)      # which commit produced the runs in this folder -> code_version.txt

## 4. Data and checkpoints (CIFAR-100 and the authors' victim checkpoints, about 200 MB)

In [ ]:
!pip -q install huggingface_hub
!python repro/download_artifacts.py --what cifar100
!python repro/prepare_cifar100.py

## 5. Arms

In [ ]:
def fb(z, w=2, extra=''):            # option (b): eta preserves head T (the paper); z = zeta = mu (x/255)
    return f'--addnoise --friends --zeta {z} --mu_rand {z} --warmup_epochs {w} {extra}'.strip()

ARMS = [
    # ---- to run now (eps 0.3) ----
    ('A_clean',          ''),                                       # shared with the a / c notebooks (runs once)
    ('B_poison',         '--addnoise'),
    # 1. eta optimizer (official: lr 100, momentum 0.9)
    ('b_z32_ref',        fb(32)),
    ('b_z32_mom0',       fb(32, extra='--friendly_momentum 0')),
    ('b_z32_lr10',       fb(32, extra='--friendly_lr 10')),
    ('b_z32_lr10_mom0',  fb(32, extra='--friendly_lr 10 --friendly_momentum 0')),
    ('b_z32_lr1000',     fb(32, extra='--friendly_lr 1000')),
    # 2. BatchNorm control: stored BatchNorm statistics never update (normalization still uses batch statistics)
    ('B_bn',             '--addnoise --bn_mode nostats'),
    ('A_bn',             '--bn_mode nostats'),
    # ---- already done (eps 0.3, official code; uncomment to re-run on this code version) ----
    # ('b_z16', fb(16)), ('b_z64', fb(64)),                          # BWT -21.4 / -19.9
    # ('b_fr_z32', '--addnoise --friends --components friendly_only --zeta 32 --warmup_epochs 2'),   # -21.6
    # ('b_rn_mu64', '--addnoise --friends --components random_only --mu_rand 64'),                  # -24.4
    # ('b_z16_ra', fb(16, extra='--random_after_warmup')),           # -21.1
]
H.run_arms(ARMS, H.base_cmd(NOISE_PKL, RESULTS_DIR, SEED), RESULTS_DIR, SEED)

## Results

In % (as in the BrainWash paper). More negative BWT = more forgetting.
`recovered_%` = how much of the BWT gap between the attack (`ours`) and the clean run an arm closes (100 = as good as clean, negative = worse than no defense).
`last_vs_attack` = last-task accuracy minus the attack's: a negative number is what the recovery costs on the new task.
`old_acc_at_eta` = old-task accuracy when the friendly noise was computed (62.0 = nothing forgotten yet).

In [ ]:
H.results_table(RESULTS_DIR, SEED)

## Per-epoch old-task accuracy (rows = epochs, columns = arms): when does the damage happen?

In [ ]:
H.per_epoch_table(RESULTS_DIR, SEED)